# RW-Causal Forest




## 1. Setup

In [ ]:
!pip install dowhy econml scikit-learn pandas numpy scipy shap openpyxl matplotlib -q

In [ ]:
import re
import time
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (average_precision_score, f1_score, recall_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              precision_recall_curve, roc_curve, auc)
from scipy.stats import wilcoxon
from econml.dml import CausalForestDML
import shap

np.random.seed(42)


In [ ]:
!pip freeze | grep -iE "^dowhy|^econml|^scikit-learn|^pandas|^numpy|^scipy|^shap|^openpyxl|^matplotlib"
import sys
print("Python version:", sys.version)


## 2. Helper Functions

In [ ]:
# 2a. Prepare one fold with no leakage: scaler and class weight fit on the
# training portion only. Optionally applies an in-fold overlap trim (School
# Type), fitting the propensity model on the training partition only and
# applying it to both sides. Groups are carried through and re-aligned after
# trimming so group-aware splitting downstream stays valid.

def weight_from(y):
    # quarter-strength class weight, same formula used throughout this
    # notebook wherever a standalone reference to it is needed
    n1 = (y == 1).sum(); n0 = (y == 0).sum()
    ratio = max(n1, n0) / min(n1, n0) if min(n1, n0) > 0 else 1.0
    return {0: 1, 1: 1 + (ratio - 1) * 0.25}

def prepare_fold(X_raw, T, Y, G, train_idx, test_idx, needs_trim=False,
                  trim_lo=0.05, trim_hi=0.95):
    X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    G_train, G_test = G[train_idx], G[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    if needs_trim:
        overlap_model = LogisticRegression(max_iter=1000)
        overlap_model.fit(X_train, T_train)
        p_train = overlap_model.predict_proba(X_train)[:, 1]
        p_test = overlap_model.predict_proba(X_test)[:, 1]
        keep_train = (p_train >= trim_lo) & (p_train <= trim_hi)
        keep_test = (p_test >= trim_lo) & (p_test <= trim_hi)
        print(f"    in-fold overlap trim | train {len(X_train)}->{keep_train.sum()} "
              f"| test {len(X_test)}->{keep_test.sum()}")
        X_train, T_train, Y_train, G_train = (X_train[keep_train], T_train[keep_train],
                                               Y_train[keep_train], G_train[keep_train])
        X_test, T_test, Y_test, G_test = (X_test[keep_test], T_test[keep_test],
                                           Y_test[keep_test], G_test[keep_test])

    n1 = (Y_train == 1).sum()
    n0 = (Y_train == 0).sum()
    ratio = max(n1, n0) / min(n1, n0) if min(n1, n0) > 0 else 1.0
    quarter_weight = {0: 1, 1: 1 + (ratio - 1) * 0.25}

    return X_train, X_test, T_train, T_test, Y_train, Y_test, G_train, G_test, quarter_weight


In [ ]:
# 2b. Fit one CausalForestDML model. LEAF_ENGINEERED = 12 per the supervisor
# ruling (validated against the full factorial search average of 12.8%
# CI-width reduction, while retaining more CATE heterogeneity than leaf=15/20).
LEAF_BASELINE = 5
LEAF_ENGINEERED = 12
WEIGHT_FRAC = 0.25

# Global log of every convergence warning raised during any fit call
# in this notebook, across every section. Report the tally, not just
# whether any occurred - a small rate on thousands of fits is a different
# finding than a large one.
import warnings
from sklearn.exceptions import ConvergenceWarning
CONVERGENCE_LOG = []
FIT_COUNT = [0]

def fit_causal_forest(X, T, Y, min_samples_leaf, class_weight=None):
    FIT_COUNT[0] += 1
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = CausalForestDML(
            model_y=LogisticRegression(class_weight=class_weight, max_iter=1000),
            model_t=RandomForestClassifier(random_state=42),
            discrete_outcome=True,
            discrete_treatment=True,
            n_estimators=500,
            min_samples_leaf=min_samples_leaf,
            max_depth=8,
            honest=True,
            random_state=42
        )
        model.fit(Y, T, X=X)
        for w in caught:
            if issubclass(w.category, ConvergenceWarning):
                CONVERGENCE_LOG.append(str(w.message))
    return model

def report_convergence_log():
    n_total = FIT_COUNT[0]
    n_warn = len(CONVERGENCE_LOG)
    if n_total == 0:
        print("Convergence check: no fit_causal_forest() calls recorded yet.")
        return
    pct = 100 * n_warn / n_total
    print(f"Convergence check: {n_warn} warning(s) across {n_total} total fit_causal_forest() calls ({pct:.2f}%)")
    if n_warn > 0:
        print("First warning text:", CONVERGENCE_LOG[0][:150])


In [ ]:
# 2c. Compute all six metrics for one fitted model
METRIC_KEYS = ["CI width", "AUC-PR", "Macro F1", "At-risk recall",
               "Equal Opportunity diff", "Equalized Odds diff"]

def evaluate_model(model, X_test, Y_test, T_test, T0=0, T1=1):
    lower, upper = model.effect_interval(X_test, T0=T0, T1=T1, alpha=0.05)
    ci_width = (upper - lower).mean()

    probs = model.models_y[0][0].predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)

    auc_pr = average_precision_score(Y_test, probs)
    macro_f1 = f1_score(Y_test, preds, average="macro")
    recall = recall_score(Y_test, preds, pos_label=1)

    group0 = (T_test == T0)
    group1 = (T_test == T1)

    tpr0 = preds[group0][Y_test[group0] == 1].mean() if (Y_test[group0] == 1).sum() > 0 else np.nan
    tpr1 = preds[group1][Y_test[group1] == 1].mean() if (Y_test[group1] == 1).sum() > 0 else np.nan
    equal_opportunity_diff = tpr1 - tpr0

    fpr0 = preds[group0][Y_test[group0] == 0].mean() if (Y_test[group0] == 0).sum() > 0 else np.nan
    fpr1 = preds[group1][Y_test[group1] == 0].mean() if (Y_test[group1] == 0).sum() > 0 else np.nan
    equalized_odds_diff = max(abs(tpr1 - tpr0), abs(fpr1 - fpr0))

    return {
        "CI width": ci_width, "AUC-PR": auc_pr, "Macro F1": macro_f1,
        "At-risk recall": recall, "Equal Opportunity diff": equal_opportunity_diff,
        "Equalized Odds diff": equalized_odds_diff,
    }


In [ ]:
# 2d. Plotting functions
BASELINE_COLOR = "#c44e52"
ENGINEERED_COLOR = "#4c72b0"

def plot_confusion_matrices(Y_true, base_preds, eng_preds, label):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    ConfusionMatrixDisplay.from_predictions(Y_true, base_preds, normalize="true",
                                             ax=axes[0], colorbar=False, cmap="Reds")
    axes[0].set_title(f"{label} - Baseline")
    ConfusionMatrixDisplay.from_predictions(Y_true, eng_preds, normalize="true",
                                             ax=axes[1], colorbar=False, cmap="Blues")
    axes[1].set_title(f"{label} - Engineered")
    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"confusion_matrix_{safe_name}.png", dpi=300)
    plt.show()

def plot_pr_roc_curves(Y_true, base_probs, eng_probs, label):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    base_p, base_r, _ = precision_recall_curve(Y_true, base_probs)
    eng_p, eng_r, _ = precision_recall_curve(Y_true, eng_probs)
    base_ap = average_precision_score(Y_true, base_probs)
    eng_ap = average_precision_score(Y_true, eng_probs)
    axes[0].plot(base_r, base_p, color=BASELINE_COLOR, label=f"Baseline (AUC-PR={base_ap:.3f})")
    axes[0].plot(eng_r, eng_p, color=ENGINEERED_COLOR, label=f"Engineered (AUC-PR={eng_ap:.3f})")
    axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
    axes[0].set_title(f"{label} - Precision-Recall")
    axes[0].legend()

    base_fpr, base_tpr, _ = roc_curve(Y_true, base_probs)
    eng_fpr, eng_tpr, _ = roc_curve(Y_true, eng_probs)
    base_auc = auc(base_fpr, base_tpr)
    eng_auc = auc(eng_fpr, eng_tpr)
    axes[1].plot(base_fpr, base_tpr, color=BASELINE_COLOR, label=f"Baseline (AUC={base_auc:.3f})")
    axes[1].plot(eng_fpr, eng_tpr, color=ENGINEERED_COLOR, label=f"Engineered (AUC={eng_auc:.3f})")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=0.8)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"{label} - ROC")
    axes[1].legend()

    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"pr_roc_curves_{safe_name}.png", dpi=300)
    plt.show()

def plot_fold_level_boxplot(baseline_values, engineered_values, label):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    data = [baseline_values, engineered_values]
    bp = ax.boxplot(data, tick_labels=["Baseline", "Engineered"],
                     showmeans=True, meanline=True, patch_artist=True)
    for patch, color in zip(bp["boxes"], [BASELINE_COLOR, ENGINEERED_COLOR]):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    for i, vals in enumerate(data, start=1):
        x = np.random.normal(i, 0.04, size=len(vals))
        ax.scatter(x, vals, alpha=0.6, s=14, color="black", zorder=3)
    ax.set_ylabel("CATE CI width")
    ax.set_title(f"{label}\n({len(baseline_values)} points per model)")
    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"fold_level_boxplot_{safe_name}.png", dpi=300)
    plt.show()


In [ ]:
# 2e. Full evaluation: GROUPED search/confirm split, GROUPED repeated CV,
# optional in-fold trim, native plotting. This is the corrected replacement
# for the earlier ungrounded (student-level) splitting.

def run_full_evaluation(X_raw, T, Y, G, label, comparisons=((0, 1),), n_repeats=10,
                         needs_trim=False):
    # Grouped, approximately-stratified 80/20 search/confirm split
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    search_idx, confirm_idx = next(outer.split(np.zeros(len(Y)), Y, groups=G))

    X_search_raw, T_search, Y_search, G_search = X_raw[search_idx], T[search_idx], Y[search_idx], G[search_idx]

    results = {c: {"baseline": {k: [] for k in METRIC_KEYS},
                    "engineered": {k: [] for k in METRIC_KEYS}} for c in comparisons}
    train_times = {"baseline": [], "engineered": []}

    for repeat in range(n_repeats):
        skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
        for tr_idx, te_idx in skf.split(np.zeros(len(Y_search)), Y_search, groups=G_search):
            X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw = prepare_fold(
                X_search_raw, T_search, Y_search, G_search, tr_idx, te_idx, needs_trim=needs_trim
            )
            if len(Y_tr) == 0 or len(Y_te) == 0 or len(set(T_tr)) < 2:
                continue

            t0 = time.time()
            baseline_model = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
            train_times["baseline"].append(time.time() - t0)

            t0 = time.time()
            engineered_model = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw)
            train_times["engineered"].append(time.time() - t0)

            for comp in comparisons:
                T0, T1 = comp
                if (T_te == T0).sum() == 0 or (T_te == T1).sum() == 0:
                    continue
                b_scores = evaluate_model(baseline_model, X_te, Y_te, T_te, T0, T1)
                e_scores = evaluate_model(engineered_model, X_te, Y_te, T_te, T0, T1)
                for k in METRIC_KEYS:
                    results[comp]["baseline"][k].append(b_scores[k])
                    results[comp]["engineered"][k].append(e_scores[k])

    print(f"===== {label}: repeated grouped cross-validation ({n_repeats} repeats x 5 folds) =====")
    for comp in comparisons:
        print(f"--- treatment {comp[1]} vs {comp[0]} ---")
        table = {}
        for k in METRIC_KEYS:
            b_vals = np.array(results[comp]["baseline"][k])
            e_vals = np.array(results[comp]["engineered"][k])
            table[k] = [f"{np.nanmean(b_vals):.4f} +/- {np.nanstd(b_vals):.4f} (n={len(b_vals)})",
                        f"{np.nanmean(e_vals):.4f} +/- {np.nanstd(e_vals):.4f} (n={len(e_vals)})"]
        print(pd.DataFrame(table, index=["Baseline", "Engineered"]).T)
        print()

    print(f"Mean training time per fit: baseline {np.mean(train_times['baseline']):.3f}s "
          f"+/- {np.std(train_times['baseline']):.3f}s | "
          f"engineered {np.mean(train_times['engineered']):.3f}s "
          f"+/- {np.std(train_times['engineered']):.3f}s")
    print()

    plot_fold_level_boxplot(
        results[comparisons[0]]["baseline"]["CI width"],
        results[comparisons[0]]["engineered"]["CI width"],
        label=label
    )

    # Statistical test computed SEPARATELY for every comparison passed in,
    # not just the first one - otherwise only the first contrast would be
    # tested, silently standing in for every contrast in this treatment.
    for comp in comparisons:
        b_ci = np.array(results[comp]["baseline"]["CI width"])
        e_ci = np.array(results[comp]["engineered"]["CI width"])
        stat, p_value = wilcoxon(b_ci, e_ci)
        diffs = e_ci - b_ci
        pooled_std = np.sqrt((b_ci.std() ** 2 + e_ci.std() ** 2) / 2)
        cohens_d_unpaired = diffs.mean() / pooled_std if pooled_std > 0 else float("nan")
        cohens_d_paired = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else float("nan")

        boot_means = [np.random.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(1000)]
        ci_lower, ci_upper = np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

        print(f"--- statistical test, treatment {comp[1]} vs {comp[0]} ---")
        print(f"Wilcoxon signed-rank test on CI width: statistic={stat:.4f}, p={p_value:.6e}")
        print(f"Cohen's d_z (paired, matches the design): {cohens_d_paired:.4f}")
        print(f"Cohen's d (unpaired, for reference only): {cohens_d_unpaired:.4f}")
        print(f"95% bootstrap CI on mean CI-width difference: [{ci_lower:.4f}, {ci_upper:.4f}]")
        print()

    # Held-out confirmation: fit once on the full (grouped) search partition
    X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw_confirm = prepare_fold(
        X_raw, T, Y, G, search_idx, confirm_idx, needs_trim=needs_trim
    )
    confirm_baseline = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
    confirm_engineered = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw_confirm)

    print(f"===== {label}: held-out confirmation (grouped, data never used in search) =====")
    for comp in comparisons:
        T0, T1 = comp
        b_conf = evaluate_model(confirm_baseline, X_te, Y_te, T_te, T0, T1)
        e_conf = evaluate_model(confirm_engineered, X_te, Y_te, T_te, T0, T1)
        print(f"--- treatment {T1} vs {T0} ---")
        print(pd.DataFrame({"Baseline": b_conf, "Engineered": e_conf}))
        print()

    b_probs = confirm_baseline.models_y[0][0].predict_proba(X_te)[:, 1]
    b_preds = (b_probs >= 0.5).astype(int)
    e_probs = confirm_engineered.models_y[0][0].predict_proba(X_te)[:, 1]
    e_preds = (e_probs >= 0.5).astype(int)

    print("Baseline confusion matrix (normalized by true class):")
    print(confusion_matrix(Y_te, b_preds, normalize="true"))
    print("Engineered confusion matrix (normalized by true class):")
    print(confusion_matrix(Y_te, e_preds, normalize="true"))

    plot_confusion_matrices(Y_te, b_preds, e_preds, label=label)
    plot_pr_roc_curves(Y_te, b_probs, e_probs, label=label)

    return confirm_baseline, confirm_engineered, X_te, Y_te, T_te


## 3. GhEduData Data

In [ ]:
ghedudata = pd.read_excel("GhEduData_Merged_Anonymized.xlsx", sheet_name="GhEduData_Merged")
print("Students loaded:", len(ghedudata))
print("Schools:", ghedudata["School_ID"].nunique())


In [ ]:
def clean_text(value):
    value = str(value).replace("\u25a1", "").strip()
    return " ".join(value.split())

messy_columns = ["Governance_Type", "Location_Category", "Avg_Class_Size", "Pct_Teachers_Degree",
                  "Teacher_Student_Ratio", "Classroom_Condition", "English_Textbook_Access",
                  "Math_Textbook_Access", "Home_Study_Access", "Socioeconomic_Profile"]

for col in messy_columns:
    ghedudata[col] = ghedudata[col].apply(clean_text)


In [ ]:
ghedudata["at_risk"] = (ghedudata["Aggregate"] >= 21).astype(int)
ghedudata["gender_num"] = ghedudata["Gender"].apply(lambda x: 1 if x == "Male" else 0)

def encode_location(value):
    levels = {"Rural": 0, "Peri-urban": 1, "Urban": 2}
    return levels.get(value, -1)

ghedudata["location_num"] = ghedudata["Location_Category"].apply(encode_location)
ghedudata["school_type_num"] = ghedudata["Governance_Type"].apply(lambda x: 1 if "Private" in x else 0)

print("At-risk count:", ghedudata["at_risk"].sum(), "out of", len(ghedudata))


In [ ]:
def encode_attendance(v):
    if "Less than 50%" in v: return 0
    if "50" in v and "74" in v: return 1
    if "75" in v and "90" in v: return 2
    if "More than 90%" in v: return 3
    return -1

def encode_homework(v):
    if "Rarely" in v: return 0
    if "Sometimes" in v: return 1
    if "Often" in v: return 2
    if "Almost always" in v: return 3
    return -1

def encode_participation(v):
    if "Passive" in v: return 0
    if "Very active" in v: return 3
    if "Moderate" in v: return 1
    if "Active" in v: return 2
    return -1

ghedudata["attendance_num"] = ghedudata["Attendance Rate"].apply(encode_attendance)
ghedudata["homework_num"] = ghedudata["Homework Submission Rate"].apply(encode_homework)
ghedudata["participation_num"] = ghedudata["Classroom Participation"].apply(encode_participation)

student_level_confounders = ["Age", "attendance_num", "homework_num", "participation_num"]


In [ ]:
def encode_class_size(v):
    sizes = {"Fewer than 20": 0, "20\u201335": 1, "36\u201350": 2,
             "51 - 65": 3, "66 - 80": 4, "More than 80": 5}
    return sizes.get(v, -1)

def encode_teacher_qual(v):
    quals = {"Less than 25%": 0, "50\u201375%": 1, "More than 75%": 2}
    return quals.get(v, -1)

def encode_teacher_ratio(v):
    if "fewer than 25" in v: return 0
    if "25\u201335" in v: return 1
    if "36\u201350" in v: return 2
    if "51 - 65" in v: return 3
    if "66 - 80" in v: return 4
    return -1

def encode_condition(v):
    if "Some classrooms are inadequate" in v: return 0
    if "Mostly adequate" in v: return 1
    if "All classrooms adequate" in v: return 2
    return -1

def encode_textbook(v):
    if "3 or more" in v: return 0
    if "between 2" in v: return 1
    if "own copy" in v: return 2
    return -1

def encode_home_study(v):
    if "Fewer than 25%" in v: return 0
    if "25\u201349%" in v: return 1
    if "More than 75%" in v: return 3
    if "50" in v: return 2
    return -1

def encode_socioeconomic(v):
    if "subsistence" in v: return 0
    if "middle income" in v: return 2
    if "high income" in v: return 3
    if "low income" in v: return 1
    return -1

ghedudata["class_size_num"] = ghedudata["Avg_Class_Size"].apply(encode_class_size)
ghedudata["teacher_qual_num"] = ghedudata["Pct_Teachers_Degree"].apply(encode_teacher_qual)
ghedudata["teacher_ratio_num"] = ghedudata["Teacher_Student_Ratio"].apply(encode_teacher_ratio)
ghedudata["classroom_condition_num"] = ghedudata["Classroom_Condition"].apply(encode_condition)
ghedudata["english_textbook_num"] = ghedudata["English_Textbook_Access"].apply(encode_textbook)
ghedudata["math_textbook_num"] = ghedudata["Math_Textbook_Access"].apply(encode_textbook)
ghedudata["home_study_num"] = ghedudata["Home_Study_Access"].apply(encode_home_study)
ghedudata["socioeconomic_num"] = ghedudata["Socioeconomic_Profile"].apply(encode_socioeconomic)

school_level_confounders = ["class_size_num", "teacher_qual_num", "teacher_ratio_num",
                             "classroom_condition_num", "english_textbook_num",
                             "math_textbook_num", "home_study_num", "socioeconomic_num"]

full_confounders = student_level_confounders + school_level_confounders

for col in ["attendance_num", "homework_num", "participation_num"] + school_level_confounders:
    unmatched = (ghedudata[col] == -1).sum()
    if unmatched > 0:
        print("WARNING - unmatched values in", col, ":", unmatched)
print("Encoding check complete.")


In [ ]:
# No school-type overlap trim here - it is now computed IN-FOLD inside
# run_full_evaluation (needs_trim=True), not upfront on the full corpus.

X_gh_full_raw = ghedudata[full_confounders].values
X_gh_student_raw = ghedudata[student_level_confounders].values

Y_gh = ghedudata["at_risk"].values
T_gh_gender = ghedudata["gender_num"].values
T_gh_location = ghedudata["location_num"].values
T_gh_schooltype = ghedudata["school_type_num"].values
G_gh = ghedudata["School_ID"].values

print("GhEduData ready:", X_gh_full_raw.shape, "| schools:", len(set(G_gh)))


## 4. GhEduData Results

In [ ]:
# 4a. Gender: grouped repeated CV + grouped held-out confirmation
gh_gender_baseline, gh_gender_engineered, X_gh_gender_confirm, Y_gh_gender_confirm, T_gh_gender_confirm = run_full_evaluation(
    X_gh_full_raw, T_gh_gender, Y_gh, G_gh, label="GhEduData - Gender", n_repeats=10
)


In [ ]:
# 4b. Location: grouped repeated CV + grouped held-out confirmation
gh_loc_baseline, gh_loc_engineered, X_gh_loc_confirm, Y_gh_loc_confirm, T_gh_loc_confirm = run_full_evaluation(
    X_gh_student_raw, T_gh_location, Y_gh, G_gh, label="GhEduData - Location",
    comparisons=((0, 1), (0, 2)), n_repeats=10
)


In [ ]:
# 4c. School type: grouped repeated CV, IN-FOLD overlap trim, grouped confirmation
gh_school_baseline, gh_school_engineered, X_gh_school_confirm, Y_gh_school_confirm, T_gh_school_confirm = run_full_evaluation(
    X_gh_student_raw, T_gh_schooltype, Y_gh, G_gh,
    label="GhEduData - School Type", n_repeats=10, needs_trim=True
)


## 5. OULAD Data


In [ ]:
url = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"
urllib.request.urlretrieve(url, "oulad.zip")
with zipfile.ZipFile("oulad.zip", "r") as zip_ref:
    zip_ref.extractall("oulad_data")

student_info = pd.read_csv("oulad_data/studentInfo.csv")
student_vle = pd.read_csv("oulad_data/studentVle.csv")
student_assessment = pd.read_csv("oulad_data/studentAssessment.csv")

print("Rows loaded:", len(student_info), "| distinct students:", student_info["id_student"].nunique())


In [ ]:
CUTOFF_DAY = 28   # chosen from the retention/leakage table, before any CATE was inspected

student_info["at_risk"] = student_info["final_result"].apply(
    lambda x: 0 if x in ["Pass", "Distinction"] else 1
)
student_info["gender_num"] = student_info["gender"].apply(lambda x: 1 if x == "M" else 0)

# DROP records with missing treatment, do not impute
n_before = len(student_info)
student_info = student_info[student_info["imd_band"].astype(str) != "?"].copy()
print(f"FILTER drop missing treatment (imd_band) | n before {n_before} | n after {len(student_info)}")

def imd_lower_bound(band_text):
    match = re.search(r"\d+", str(band_text))
    return int(match.group())

student_info["imd_lower_bound"] = student_info["imd_band"].apply(imd_lower_bound)

def group_deprivation(lower_bound):
    if lower_bound <= 20:
        return 0
    elif lower_bound <= 60:
        return 1
    else:
        return 2

student_info["location_num"] = student_info["imd_lower_bound"].apply(group_deprivation)


In [ ]:
def encode_education(level):
    levels = {"No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
              "HE Qualification": 3, "Post Graduate Qualification": 4}
    return levels.get(level, -1)

student_info["education_num"] = student_info["highest_education"].apply(encode_education)

def encode_age(band):
    bands = {"0-35": 0, "35-55": 1, "55<=": 2}
    return bands.get(band, -1)

student_info["age_num"] = student_info["age_band"].apply(encode_age)

# Engagement window applied BEFORE aggregation, merge on the FULL
# presentation key (not id_student alone)
vle_windowed = student_vle[student_vle["date"] <= CUTOFF_DAY]
print(f"FILTER engagement window date<={CUTOFF_DAY} | n before {len(student_vle)} | n after {len(vle_windowed)}")

assess_windowed = student_assessment[student_assessment["date_submitted"] <= CUTOFF_DAY]
print(f"FILTER submissions date_submitted<={CUTOFF_DAY} | n before {len(student_assessment)} | n after {len(assess_windowed)}")

engagement = vle_windowed.groupby(["id_student", "code_module", "code_presentation"]).agg(
    total_clicks=("sum_click", "sum"),
    active_days=("date", "nunique")
).reset_index()

submitted = assess_windowed.groupby("id_student").size().reset_index(name="submitted_count")

oulad_data = student_info.merge(engagement, on=["id_student", "code_module", "code_presentation"], how="left")
print(f"MERGE engagement (full presentation key) | n before {len(student_info)} | n after {len(oulad_data)}")
oulad_data = oulad_data.merge(submitted, on="id_student", how="left")
print(f"MERGE submissions | n before {len(student_info)} | n after {len(oulad_data)}")

oulad_data["total_clicks"] = oulad_data["total_clicks"].fillna(0)
oulad_data["active_days"] = oulad_data["active_days"].fillna(0)
oulad_data["submitted_count"] = oulad_data["submitted_count"].fillna(0)

print("Any unmatched education or age values?",
      (oulad_data["education_num"] == -1).sum(), (oulad_data["age_num"] == -1).sum())


In [ ]:
oulad_confounders = ["education_num", "age_num", "total_clicks", "active_days", "submitted_count"]

X_oulad_raw = oulad_data[oulad_confounders].values
T_oulad_gender = oulad_data["gender_num"].values
T_oulad_location = oulad_data["location_num"].values
Y_oulad = oulad_data["at_risk"].values
G_oulad = oulad_data["id_student"].values   # grouped by STUDENT, not by row

print("OULAD ready:", X_oulad_raw.shape, "| at-risk rate:", Y_oulad.mean().round(3),
      "| distinct students:", len(set(G_oulad)))


## 6. OULAD Results

In [ ]:
# 6a. Gender: grouped repeated CV + grouped held-out confirmation
oulad_gender_baseline, oulad_gender_engineered, X_oulad_gender_confirm, Y_oulad_gender_confirm, T_oulad_gender_confirm = run_full_evaluation(
    X_oulad_raw, T_oulad_gender, Y_oulad, G_oulad, label="OULAD - Gender", n_repeats=5
)


In [ ]:
# 6b. Location: grouped repeated CV + grouped held-out confirmation
oulad_loc_baseline, oulad_loc_engineered, X_oloc_confirm, Y_oloc_confirm, T_oloc_confirm = run_full_evaluation(
    X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, label="OULAD - Location",
    comparisons=((0, 1), (0, 2)), n_repeats=5
)


## 7. Ablation Isolation

In [ ]:
# 7a. Isolate each engineered component separately, GhEduData Gender,
# using the grouped, corrected splitting protocol throughout.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
search_idx, confirm_idx = next(outer.split(np.zeros(len(Y_gh)), Y_gh, groups=G_gh))
X_search_raw = X_gh_full_raw[search_idx]
T_search = T_gh_gender[search_idx]
Y_search = Y_gh[search_idx]
G_search = G_gh[search_idx]

ablation_results = {c: {k: [] for k in METRIC_KEYS} for c in ["baseline", "leaf_only", "weight_only", "full"]}
# Track model_y's residual variance for baseline and weight_only, per
# fold, to test whether the weight component's CI-width cost correlates
# with inflated outcome-model residual variance (a candidate mechanism).
resid_var_baseline, resid_var_weight = [], []

for repeat in range(10):
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for tr_idx, te_idx in skf.split(np.zeros(len(Y_search)), Y_search, groups=G_search):
        X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw = prepare_fold(
            X_search_raw, T_search, Y_search, G_search, tr_idx, te_idx
        )
        if len(Y_tr) == 0 or len(Y_te) == 0 or len(set(T_tr)) < 2:
            continue

        m_baseline = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
        m_leaf_only = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED)
        m_weight_only = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE, class_weight=qw)
        m_full = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw)

        for name, model in [("baseline", m_baseline), ("leaf_only", m_leaf_only),
                             ("weight_only", m_weight_only), ("full", m_full)]:
            scores = evaluate_model(model, X_te, Y_te, T_te)
            for k in METRIC_KEYS:
                ablation_results[name][k].append(scores[k])

        p_base = m_baseline.models_y[0][0].predict_proba(X_te)[:, 1]
        p_wt = m_weight_only.models_y[0][0].predict_proba(X_te)[:, 1]
        resid_var_baseline.append(np.var(Y_te - p_base))
        resid_var_weight.append(np.var(Y_te - p_wt))

print("=== Ablation isolation: GhEduData - Gender (grouped, 10 repeats x 5 folds) ===")
table = {}
for name in ["baseline", "leaf_only", "weight_only", "full"]:
    table[name] = [f"{np.nanmean(ablation_results[name][k]):.4f} +/- {np.nanstd(ablation_results[name][k]):.4f}"
                   for k in METRIC_KEYS]
ablation_table = pd.DataFrame(table, index=METRIC_KEYS)
print(ablation_table)


In [ ]:
# 7a2. Test whether the weight component's CI-width cost correlates
# with inflated outcome-model residual variance - a candidate mechanism
# for the +0.0056 finding, not previously tested on the corrected protocol.
from scipy.stats import pearsonr, spearmanr

resid_var_baseline = np.array(resid_var_baseline)
resid_var_weight = np.array(resid_var_weight)
resid_var_delta = resid_var_weight - resid_var_baseline

ci_width_base_arr = np.array(ablation_results["baseline"]["CI width"])
ci_width_wt_arr = np.array(ablation_results["weight_only"]["CI width"])
ci_width_delta = ci_width_wt_arr - ci_width_base_arr

assert len(resid_var_delta) == len(ci_width_delta), "fold counts must match"

r_pearson, p_pearson = pearsonr(resid_var_delta, ci_width_delta)
r_spearman, p_spearman = spearmanr(resid_var_delta, ci_width_delta)

print("=== Residual-variance mechanism test, GhEduData Gender ===")
print(f"n folds = {len(resid_var_delta)}")
print(f"Mean residual-variance delta (weight_only - baseline): {resid_var_delta.mean():.5f} +/- {resid_var_delta.std():.5f}")
print(f"Mean CI-width delta (weight_only - baseline): {ci_width_delta.mean():.5f} +/- {ci_width_delta.std():.5f}")
print(f"Pearson r = {r_pearson:.4f} (p={p_pearson:.4f})")
print(f"Spearman r = {r_spearman:.4f} (p={p_spearman:.4f})")
print()
if abs(r_pearson) < 0.3 and abs(r_spearman) < 0.3:
    print("Weak correlation - the residual-variance mechanism is NOT supported")
    print("as an explanation for the weight component's CI-width cost.")
else:
    print("Correlation is not weak - report both coefficients and interpret")
    print("with the fold count above in mind before treating this as confirmed.")


In [ ]:
# 7b. Plot the ablation comparison for the primary metric (CI width)
fig, ax = plt.subplots(figsize=(6, 4.5))
conditions = ["baseline", "leaf_only", "weight_only", "full"]
means = [np.nanmean(ablation_results[c]["CI width"]) for c in conditions]
sds = [np.nanstd(ablation_results[c]["CI width"]) for c in conditions]
colors = ["#c44e52", "#dd8452", "#8172b3", "#4c72b0"]

ax.bar(conditions, means, yerr=sds, capsize=5, color=colors, alpha=0.8)
ax.set_ylabel("CATE CI width (mean +/- SD)")
ax.set_title("GhEduData - Gender: ablation isolation (leaf=12)")
plt.tight_layout()
plt.savefig("ablation_ci_width_barplot.png", dpi=300)
plt.show()


### 7c. OULAD Ablation Isolation


In [ ]:
outer_o = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
search_idx_o, confirm_idx_o = next(outer_o.split(np.zeros(len(Y_oulad)), Y_oulad, groups=G_oulad))
X_search_raw_o = X_oulad_raw[search_idx_o]
T_search_o = T_oulad_gender[search_idx_o]
Y_search_o = Y_oulad[search_idx_o]
G_search_o = G_oulad[search_idx_o]

oulad_ablation_results = {c: {k: [] for k in METRIC_KEYS} for c in ["baseline", "leaf_only", "weight_only", "full"]}

for repeat in range(5):
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for tr_idx, te_idx in skf.split(np.zeros(len(Y_search_o)), Y_search_o, groups=G_search_o):
        X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw = prepare_fold(
            X_search_raw_o, T_search_o, Y_search_o, G_search_o, tr_idx, te_idx
        )
        if len(Y_tr) == 0 or len(Y_te) == 0 or len(set(T_tr)) < 2:
            continue

        m_baseline = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
        m_leaf_only = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED)
        m_weight_only = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE, class_weight=qw)
        m_full = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw)

        for name, model in [("baseline", m_baseline), ("leaf_only", m_leaf_only),
                             ("weight_only", m_weight_only), ("full", m_full)]:
            scores = evaluate_model(model, X_te, Y_te, T_te)
            for k in METRIC_KEYS:
                oulad_ablation_results[name][k].append(scores[k])

print("=== Ablation isolation: OULAD - Gender (grouped, 5 repeats x 5 folds) ===")
table_o = {}
for name in ["baseline", "leaf_only", "weight_only", "full"]:
    table_o[name] = [f"{np.nanmean(oulad_ablation_results[name][k]):.4f} +/- {np.nanstd(oulad_ablation_results[name][k]):.4f}"
                     for k in METRIC_KEYS]
oulad_ablation_table = pd.DataFrame(table_o, index=METRIC_KEYS)
print(oulad_ablation_table)
print()
print("Compare against GhEduData Gender's ablation_results above: if the")
print("SAME pattern holds (baseline == leaf_only on AUC-PR/F1/recall/fairness,")
print("weight_only == full on the same four), that is real, reported evidence")
print("component-separation claim for OULAD, replacing the assertion.")


### 7d. Refutation Tests


In [ ]:
outer_r = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
search_idx_r, _ = next(outer_r.split(np.zeros(len(Y_gh)), Y_gh, groups=G_gh))
X_r_raw = X_gh_full_raw[search_idx_r]
T_r = T_gh_gender[search_idx_r]
Y_r = Y_gh[search_idx_r]

sc_r = StandardScaler()
X_r = sc_r.fit_transform(X_r_raw)
cw_r = weight_from(Y_r)

def fit_and_get_ate(X_in, T_in, Y_in, class_weight):
    m = fit_causal_forest(X_in, T_in, Y_in, min_samples_leaf=LEAF_ENGINEERED, class_weight=class_weight)
    return float(np.asarray(m.ate(X_in, T0=0, T1=1)).ravel()[0])

original_ate = fit_and_get_ate(X_r, T_r, Y_r, cw_r)
print("=== Refutation tests, GhEduData Gender (engineered model) ===")
print(f"Original ATE: {original_ate:.4f}")
print()

# 1. Placebo treatment: real T replaced with a random permutation, breaking
# any true T-Y relationship - the ATE should collapse toward zero.
T_placebo = np.random.RandomState(0).permutation(T_r)
cw_placebo = weight_from(Y_r)
placebo_ate = fit_and_get_ate(X_r, T_placebo, Y_r, cw_placebo)
print(f"1. PLACEBO TREATMENT: ATE = {placebo_ate:.4f} (expect near zero, vs original {original_ate:.4f})")

# 2. Random common cause: one irrelevant random confounder added to X - a
# genuine causal estimate should barely move.
X_r_extra = np.column_stack([X_r, np.random.RandomState(1).randn(len(Y_r))])
extra_ate = fit_and_get_ate(X_r_extra, T_r, Y_r, cw_r)
print(f"2. RANDOM COMMON CAUSE: ATE = {extra_ate:.4f} (expect close to {original_ate:.4f})")

# 3. Data subset: refit on five random 80% subsets - a genuine estimate
# should stay reasonably stable in sign and rough magnitude.
subset_ates = []
for seed in range(5):
    rng = np.random.RandomState(seed + 10)
    idx = rng.choice(len(Y_r), size=int(len(Y_r) * 0.8), replace=False)
    if len(set(T_r[idx])) < 2:
        continue
    cw_sub = weight_from(Y_r[idx])
    subset_ates.append(fit_and_get_ate(X_r[idx], T_r[idx], Y_r[idx], cw_sub))
print(f"3. DATA SUBSET (5 refits, 80% each): ATEs = {[round(a,4) for a in subset_ates]}")
print(f"   mean = {np.mean(subset_ates):.4f}, sd = {np.std(subset_ates):.4f} (expect stable sign, roughly near {original_ate:.4f})")

print()
print("PASS/FAIL is a judgement call, not automated here - report all four")
print("numbers above plainly. A placebo ATE far from zero, or subset ATEs")
print("that flip sign, are the specific patterns that would undermine the")
print("causal claim; report honestly whichever pattern is actually observed.")


## 8. SHAP Equity Diagnostics

In [ ]:
def run_shap_diagnostics(model, X_confirm, feature_names, label):
    shap_dict = model.shap_values(X_confirm, feature_names=feature_names)
    outcome_key = list(shap_dict.keys())[0]
    treatment_key = list(shap_dict[outcome_key].keys())[0]
    explanation = shap_dict[outcome_key][treatment_key]

    mean_abs_shap = np.abs(explanation.values).mean(axis=0)
    ranking = sorted(zip(explanation.feature_names, mean_abs_shap), key=lambda x: -x[1])

    print(f"=== SHAP ranking: {label} ===")
    for name, value in ranking:
        print(f"{name}: {value:.4f}")
    print()

    fig, ax = plt.subplots(figsize=(6, max(2.5, 0.35 * len(ranking))))
    names = [r[0] for r in ranking][::-1]
    values = [r[1] for r in ranking][::-1]
    ax.barh(names, values, color="#4c72b0")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(label)
    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"shap_ranking_{safe_name}.png", dpi=300)
    plt.show()

    return explanation


In [ ]:
gh_gender_shap = run_shap_diagnostics(
    gh_gender_engineered, X_gh_gender_confirm, full_confounders, label="GhEduData - Gender"
)


In [ ]:
gh_loc_shap = run_shap_diagnostics(
    gh_loc_engineered, X_gh_loc_confirm, student_level_confounders, label="GhEduData - Location"
)


In [ ]:
gh_school_shap = run_shap_diagnostics(
    gh_school_engineered, X_gh_school_confirm, student_level_confounders, label="GhEduData - School Type"
)


In [ ]:
oulad_gender_shap = run_shap_diagnostics(
    oulad_gender_engineered, X_oulad_gender_confirm, oulad_confounders, label="OULAD - Gender"
)


In [ ]:
oulad_loc_shap = run_shap_diagnostics(
    oulad_loc_engineered, X_oloc_confirm, oulad_confounders, label="OULAD - Location"
)


## 9. Dual-Model Fairness Reporting


In [ ]:
!pip install fairlearn -q

In [ ]:
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

DECISION_THR = 0.5

def fairness_gap(y_true, y_pred, t, t0, t1):
    tpr = {}
    for tv in (t0, t1):
        mask = (t == tv) & (y_true == 1)
        tpr[tv] = y_pred[mask].mean() if mask.sum() > 0 else np.nan
    return tpr[t1] - tpr[t0]

def run_dual_model_fairness(X_raw_full, T_full, Y_full, G_full, label, t0, t1, needs_trim=False, n_repeats=10):
    """Compares the causal forest's own model_y against a separately-fit
    fairness-constrained classifier, on identical training data. Returns
    both arms' recall/F1/fairness-gap so an adoption decision can be made
    per treatment, not assumed."""
    keep = np.isin(T_full, [t0, t1])
    X_raw, T, Yv, Gv = X_raw_full[keep], T_full[keep], Y_full[keep], G_full[keep]

    results = {"current_model_y": {"gap": [], "recall": [], "f1": []},
               "separate_classifier": {"gap": [], "recall": [], "f1": []}}

    for repeat in range(n_repeats):
        skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
        for tr, te in skf.split(np.zeros(len(Yv)), Yv, groups=Gv):
            X_tr_raw, X_te_raw = X_raw[tr], X_raw[te]
            T_tr, T_te = T[tr], T[te]
            Y_tr, Y_te = Yv[tr], Yv[te]

            if needs_trim:
                sc_ov = StandardScaler()
                X_tr_s = sc_ov.fit_transform(X_tr_raw)
                ov = LogisticRegression(max_iter=1000).fit(X_tr_s, T_tr)
                p_tr = ov.predict_proba(X_tr_s)[:, 1]
                keep_tr = (p_tr >= 0.05) & (p_tr <= 0.95)
                X_tr_raw, T_tr, Y_tr = X_tr_raw[keep_tr], T_tr[keep_tr], Y_tr[keep_tr]

            if len(set(T_tr)) < 2:
                continue
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr_raw)
            X_te = sc.transform(X_te_raw)
            cw = weight_from(Y_tr)

            m = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=cw)
            p_current = m.models_y[0][0].predict_proba(X_te)[:, 1]
            pred_current = (p_current >= DECISION_THR).astype(int)
            results["current_model_y"]["gap"].append(fairness_gap(Y_te, pred_current, T_te, t0, t1))
            results["current_model_y"]["recall"].append(recall_score(Y_te, pred_current))
            results["current_model_y"]["f1"].append(f1_score(Y_te, pred_current, average="macro"))

            mitigator = ExponentiatedGradient(LogisticRegression(class_weight=cw, max_iter=1000),
                                              constraints=EqualizedOdds())
            mitigator.fit(X_tr, Y_tr, sensitive_features=T_tr)
            # ExponentiatedGradient.predict() is randomized BY DESIGN (see its
            # own docstring) and depends on the global numpy random state if
            # no seed is passed - this made results genuinely irreproducible
            # across separate sessions. Fixed by averaging K seeded draws per
            # fold, smoothing the algorithm's inherent randomness rather than
            # depending on one uncontrolled coin flip.
            K_DRAWS = 10
            pred_probs = np.mean([mitigator.predict(X_te, random_state=1000 + k) for k in range(K_DRAWS)], axis=0)
            pred_sep = (pred_probs >= 0.5).astype(int)
            results["separate_classifier"]["gap"].append(fairness_gap(Y_te, pred_sep, T_te, t0, t1))
            results["separate_classifier"]["recall"].append(recall_score(Y_te, pred_sep))
            results["separate_classifier"]["f1"].append(f1_score(Y_te, pred_sep, average="macro"))

        print(f"  [{label}] repeat {repeat} done")

    summary = {}
    for approach in results:
        gap = np.array(results[approach]["gap"], dtype=float)
        recall = np.array(results[approach]["recall"], dtype=float)
        f1v = np.array(results[approach]["f1"], dtype=float)
        summary[approach] = dict(mean_gap=np.nanmean(gap), mean_abs_gap=np.nanmean(np.abs(gap)),
                                 mean_recall=np.nanmean(recall), sd_recall=np.nanstd(recall),
                                 mean_f1=np.nanmean(f1v), sd_f1=np.nanstd(f1v))
    return summary


In [ ]:
# --- Run and report, per treatment/contrast, with an explicit adoption verdict ---
# GhEduData: 10 repeats, matching its established protocol throughout.
# OULAD: 5 repeats, matching ITS established protocol throughout (M14) -
# extended here per Objective 3's cross-dataset validation requirement:
# the fairness-mitigation approach, like the CI-width engineering itself,
# needs checking on OULAD, not assumed to transfer.
dual_model_results = {}
dual_model_results["Gender"] = run_dual_model_fairness(X_gh_full_raw, T_gh_gender, Y_gh, G_gh, "Gender", 0, 1, n_repeats=10)
dual_model_results["Location_1v0"] = run_dual_model_fairness(X_gh_student_raw, T_gh_location, Y_gh, G_gh, "Location_1v0", 0, 1, n_repeats=10)
dual_model_results["Location_2v0"] = run_dual_model_fairness(X_gh_student_raw, T_gh_location, Y_gh, G_gh, "Location_2v0", 0, 2, n_repeats=10)
dual_model_results["SchoolType"] = run_dual_model_fairness(X_gh_student_raw, T_gh_schooltype, Y_gh, G_gh, "SchoolType", 0, 1, needs_trim=True, n_repeats=10)
dual_model_results["OULAD_Gender"] = run_dual_model_fairness(X_oulad_raw, T_oulad_gender, Y_oulad, G_oulad, "OULAD_Gender", 0, 1, n_repeats=5)
dual_model_results["OULAD_Location_1v0"] = run_dual_model_fairness(X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, "OULAD_Location_1v0", 0, 1, n_repeats=5)
dual_model_results["OULAD_Location_2v0"] = run_dual_model_fairness(X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, "OULAD_Location_2v0", 0, 2, n_repeats=5)

print("\n" + "=" * 90)
print("RESULT - dual-model fairness reporting (GhEduData: 10 repeats x 5 folds; OULAD: 5 repeats x 5 folds)")
print("=" * 90)
print(f"{'Treatment':<20}{'Approach':<22}{'mean_abs_gap':>14}{'recall (mean+/-SD)':>22}{'F1 (mean+/-SD)':>20}")
for label, summ in dual_model_results.items():
    for approach in ["current_model_y", "separate_classifier"]:
        r = summ[approach]
        recall_str = f"{r['mean_recall']:.4f}+/-{r['sd_recall']:.4f}"
        f1_str = f"{r['mean_f1']:.4f}+/-{r['sd_f1']:.4f}"
        print(f"{label:<20}{approach:<22}{r['mean_abs_gap']:>14.4f}{recall_str:>22}{f1_str:>20}")

print("\nADOPTION VERDICT (both gap must narrow AND recall must improve or hold):")
adopted = {}
for label, summ in dual_model_results.items():
    c, s = summ["current_model_y"], summ["separate_classifier"]
    gap_improves = s["mean_abs_gap"] < c["mean_abs_gap"]
    recall_improves = s["mean_recall"] >= c["mean_recall"]
    adopted[label] = gap_improves and recall_improves
    verdict = "ADOPT separate_classifier" if adopted[label] else "KEEP current_model_y (not adopted)"
    print(f"  {label:<20}: {verdict}")

print("\nRecall/F1 rows and confusion matrices should be regenerated")
print("from 'separate_classifier' for every treatment marked ADOPT above, and left")
print("as model_y's own predictions for any treatment marked KEEP.")


### 9b. Confirmation-Partition Predictions


In [ ]:
def confirm_both_approaches(X_raw_full, T_full, Y_full, G_full, t0, t1, needs_trim=False):
    """Computes BOTH model_y's own predictions AND the separate fairness
    classifier's predictions on the confirmation partition, for EVERY
    treatment - not just the ones the search partition favoured. This
    lets the search-based adoption decision be checked against genuinely
    unseen data, rather than assumed to hold there."""
    keep = np.isin(T_full, [t0, t1])
    X_raw, T, Yv, Gv = X_raw_full[keep], T_full[keep], Y_full[keep], G_full[keep]

    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    search_idx, confirm_idx = next(outer.split(np.zeros(len(Yv)), Yv, groups=Gv))
    X_search_raw, T_search, Y_search = X_raw[search_idx], T[search_idx], Yv[search_idx]
    X_confirm_raw, T_confirm, Y_confirm = X_raw[confirm_idx], T[confirm_idx], Yv[confirm_idx]

    if needs_trim:
        sc_ov = StandardScaler()
        X_search_s = sc_ov.fit_transform(X_search_raw)
        ov = LogisticRegression(max_iter=1000).fit(X_search_s, T_search)
        p_search = ov.predict_proba(X_search_s)[:, 1]
        keep_search = (p_search >= 0.05) & (p_search <= 0.95)
        X_search_raw, T_search, Y_search = X_search_raw[keep_search], T_search[keep_search], Y_search[keep_search]

    sc = StandardScaler()
    X_search = sc.fit_transform(X_search_raw)
    X_confirm = sc.transform(X_confirm_raw)
    cw = weight_from(Y_search)

    # Approach 1: model_y, via the real causal forest (untouched, feeds CATE)
    m = fit_causal_forest(X_search, T_search, Y_search, min_samples_leaf=LEAF_ENGINEERED, class_weight=cw)
    p_model_y = m.models_y[0][0].predict_proba(X_confirm)[:, 1]
    pred_model_y = (p_model_y >= 0.5).astype(int)

    # Approach 2: separate fairness-constrained classifier
    mitigator = ExponentiatedGradient(LogisticRegression(class_weight=cw, max_iter=1000),
                                      constraints=EqualizedOdds())
    mitigator.fit(X_search, Y_search, sensitive_features=T_search)
    K_DRAWS = 10
    pred_probs = np.mean([mitigator.predict(X_confirm, random_state=2000 + k) for k in range(K_DRAWS)], axis=0)
    pred_separate = (pred_probs >= 0.5).astype(int)

    def summarize(pred):
        return dict(recall=recall_score(Y_confirm, pred),
                    f1=f1_score(Y_confirm, pred, average="macro"),
                    confusion=confusion_matrix(Y_confirm, pred, normalize="true"))

    return dict(model_y=summarize(pred_model_y), separate_classifier=summarize(pred_separate))

print("=" * 90)
print("CONFIRMATION-PARTITION RESULTS - BOTH APPROACHES, EVERY TREATMENT")
print("Checks whether the search-partition adoption decision holds on data")
print("neither approach has ever touched. ADOPT/KEEP above was decided on")
print("search-partition data alone; this is an independent check, not a")
print("re-derivation of that decision.")
print("=" * 90)

confirm_results = {}
confirm_results["Gender"] = confirm_both_approaches(X_gh_full_raw, T_gh_gender, Y_gh, G_gh, 0, 1)
confirm_results["Location_1v0"] = confirm_both_approaches(X_gh_student_raw, T_gh_location, Y_gh, G_gh, 0, 1)
confirm_results["Location_2v0"] = confirm_both_approaches(X_gh_student_raw, T_gh_location, Y_gh, G_gh, 0, 2)
confirm_results["SchoolType"] = confirm_both_approaches(X_gh_student_raw, T_gh_schooltype, Y_gh, G_gh, 0, 1, needs_trim=True)
confirm_results["OULAD_Gender"] = confirm_both_approaches(X_oulad_raw, T_oulad_gender, Y_oulad, G_oulad, 0, 1)
confirm_results["OULAD_Location_1v0"] = confirm_both_approaches(X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, 0, 1)
confirm_results["OULAD_Location_2v0"] = confirm_both_approaches(X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, 0, 2)

for label, r in confirm_results.items():
    print(f"\n{label}:")
    for approach in ["model_y", "separate_classifier"]:
        rr = r[approach]
        print(f"  {approach:<20} recall = {rr['recall']:.4f} | macro F1 = {rr['f1']:.4f}")
        print(f"  {'':<20} confusion matrix (normalised by true class):\n{rr['confusion']}")


### 9c. Multiple Confirmation Splits


In [ ]:
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

def fairness_gap(y_true, y_pred, t, t0, t1):
    # same definition as Section 9 - repeated here so 9c is fully
    # self-contained and does not implicitly depend on Section 9's
    # own cell having been run first in this session.
    tpr = {}
    for tv in (t0, t1):
        mask = (t == tv) & (y_true == 1)
        tpr[tv] = y_pred[mask].mean() if mask.sum() > 0 else np.nan
    return tpr[t1] - tpr[t0]

N_CONFIRM_SPLITS = 5

def confirm_one_split(X_raw_full, T_full, Y_full, G_full, t0, t1, seed, needs_trim=False):
    keep = np.isin(T_full, [t0, t1])
    X_raw, T, Yv, Gv = X_raw_full[keep], T_full[keep], Y_full[keep], G_full[keep]

    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    search_idx, confirm_idx = next(outer.split(np.zeros(len(Yv)), Yv, groups=Gv))
    X_search_raw, T_search, Y_search = X_raw[search_idx], T[search_idx], Yv[search_idx]
    X_confirm_raw, T_confirm, Y_confirm = X_raw[confirm_idx], T[confirm_idx], Yv[confirm_idx]

    if needs_trim:
        sc_ov = StandardScaler()
        X_search_s = sc_ov.fit_transform(X_search_raw)
        ov = LogisticRegression(max_iter=1000).fit(X_search_s, T_search)
        p_search = ov.predict_proba(X_search_s)[:, 1]
        keep_search = (p_search >= 0.05) & (p_search <= 0.95)
        X_search_raw, T_search, Y_search = X_search_raw[keep_search], T_search[keep_search], Y_search[keep_search]

    if len(set(T_search)) < 2 or len(set(T_confirm)) < 2:
        return None

    sc = StandardScaler()
    X_search = sc.fit_transform(X_search_raw)
    X_confirm = sc.transform(X_confirm_raw)
    cw = weight_from(Y_search)

    m = fit_causal_forest(X_search, T_search, Y_search, min_samples_leaf=LEAF_ENGINEERED, class_weight=cw)
    p_model_y = m.models_y[0][0].predict_proba(X_confirm)[:, 1]
    pred_model_y = (p_model_y >= 0.5).astype(int)

    mitigator = ExponentiatedGradient(LogisticRegression(class_weight=cw, max_iter=1000),
                                      constraints=EqualizedOdds())
    mitigator.fit(X_search, Y_search, sensitive_features=T_search)
    pred_probs = np.mean([mitigator.predict(X_confirm, random_state=3000 + seed * 100 + k) for k in range(10)], axis=0)
    pred_separate = (pred_probs >= 0.5).astype(int)

    return dict(my_recall=recall_score(Y_confirm, pred_model_y),
                my_f1=f1_score(Y_confirm, pred_model_y, average="macro"),
                my_gap=abs(fairness_gap(Y_confirm, pred_model_y, T_confirm, t0, t1)),
                sep_recall=recall_score(Y_confirm, pred_separate),
                sep_f1=f1_score(Y_confirm, pred_separate, average="macro"),
                sep_gap=abs(fairness_gap(Y_confirm, pred_separate, T_confirm, t0, t1)))

MAX_SEED_ATTEMPTS = 30  # generous cap; some treatments (esp. Location)
                        # have small enough subgroups that a single random
                        # split can easily lack one treatment level entirely

def multi_confirm(X_raw_full, T_full, Y_full, G_full, label, t0, t1, needs_trim=False):
    """Keeps trying new seeds until N_CONFIRM_SPLITS succeed, rather than
    giving up after exactly N_CONFIRM_SPLITS attempts. A treatment needing
    many more attempts than others to reach that count is itself evidence
    of how fragile its evidence base is - reported, not hidden."""
    runs = []
    seed = 0
    attempts = 0
    while len(runs) < N_CONFIRM_SPLITS and attempts < MAX_SEED_ATTEMPTS:
        r = confirm_one_split(X_raw_full, T_full, Y_full, G_full, t0, t1, seed, needs_trim)
        attempts += 1
        if r is not None:
            runs.append(r)
            print(f"  [{label}] confirmation split {len(runs)-1} done (seed {seed})")
        else:
            print(f"  [{label}] seed {seed} skipped (a partition lacked both treatment levels)")
        seed += 1
    if len(runs) < N_CONFIRM_SPLITS:
        print(f"  [{label}] WARNING: only {len(runs)} of {N_CONFIRM_SPLITS} splits succeeded "
              f"after {attempts} attempts - report this n, do not silently average fewer.")
    summary = {"n_successful": len(runs), "n_attempts": attempts}
    for k in ["my_recall", "my_f1", "my_gap", "sep_recall", "sep_f1", "sep_gap"]:
        vals = np.array([r[k] for r in runs], dtype=float)
        n_valid = int(np.sum(~np.isnan(vals)))
        if n_valid > 0:
            # nanmean/nanstd: a gap can be NaN in one split (a treatment
            # group had zero actual positives there) without discarding
            # the other splits' perfectly valid values - plain mean/std
            # would silently propagate that single NaN across everything.
            summary[k] = (np.nanmean(vals), np.nanstd(vals), n_valid)
        else:
            summary[k] = (float("nan"), float("nan"), 0)
    return summary

print("=" * 90)
print(f"MULTIPLE CONFIRMATION SPLITS ({N_CONFIRM_SPLITS} splits), ALL TREATMENTS")
print("Tests whether the search-based verdict is corroborated by the AVERAGE")
print("of several small confirmation checks, rather than any single one.")
print("=" * 90)

multi_confirm_results = {}
multi_confirm_results["Gender"] = multi_confirm(X_gh_full_raw, T_gh_gender, Y_gh, G_gh, "Gender", 0, 1)
multi_confirm_results["Location_1v0"] = multi_confirm(X_gh_student_raw, T_gh_location, Y_gh, G_gh, "Location_1v0", 0, 1)
multi_confirm_results["Location_2v0"] = multi_confirm(X_gh_student_raw, T_gh_location, Y_gh, G_gh, "Location_2v0", 0, 2)
multi_confirm_results["SchoolType"] = multi_confirm(X_gh_student_raw, T_gh_schooltype, Y_gh, G_gh, "SchoolType", 0, 1, needs_trim=True)
multi_confirm_results["OULAD_Gender"] = multi_confirm(X_oulad_raw, T_oulad_gender, Y_oulad, G_oulad, "OULAD_Gender", 0, 1)
multi_confirm_results["OULAD_Location_1v0"] = multi_confirm(X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, "OULAD_Location_1v0", 0, 1)
multi_confirm_results["OULAD_Location_2v0"] = multi_confirm(X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, "OULAD_Location_2v0", 0, 2)

print("\n" + "=" * 90)
print(f"RESULT - averaged over {N_CONFIRM_SPLITS} confirmation splits")
print("=" * 90)
col_w = 16
print(f"{'Treatment':<20}{'n':>6}"
      + "".join(f"{h:>{col_w}}" for h in ["my_recall", "my_F1", "my_gap", "sep_recall", "sep_F1", "sep_gap"]))
for label, s in multi_confirm_results.items():
    n_str = f"{s['n_successful']}/{s['n_attempts']}"
    vals = []
    for k in ["my_recall", "my_f1", "my_gap", "sep_recall", "sep_f1", "sep_gap"]:
        mean, std, n_valid = s[k]
        # gap can be valid on fewer splits than recall/F1 (a treatment
        # group had zero actual positives in some splits) - show that
        # explicitly rather than let it look like the full split count.
        suffix = f" (n={n_valid})" if "gap" in k and n_valid != s["n_successful"] else ""
        vals.append(f"{mean:.4f}+/-{std:.3f}{suffix}")
    print(f"{label:<20}{n_str:>6}" + "".join(f"{v:>{col_w}}" for v in vals))

print("\nCompare each row's separate vs model_y numbers against the Section 9")
print("ADOPTION VERDICT above. If they now agree more often than the single-split")
print("check in Section 9b did, the instability was mostly about the confirmation")
print("check itself, not the underlying method.")


## 10. Subgroup Identification


In [ ]:
from scipy import stats

ALPHA = 0.05

N_SPLITS_SUBGROUP = 5
TRIM_LO, TRIM_HI = 0.05, 0.95
MIN_SUBGROUP_N = 20
FDR_Q = 0.05

def cluster_robust_mean(tau, clusters):
    """Mean effect with a variance clustered on school. Does NOT carry the
    forest's own estimation uncertainty - state that when reporting it."""
    tau = np.asarray(tau, float)
    n = len(tau)
    m = tau.mean()
    codes = pd.factorize(clusters)[0]
    k = len(np.unique(codes))
    if k < 2 or n < 2:
        return m, np.nan, np.nan, np.nan, k
    resid_sums = np.array([(tau[codes == c] - m).sum() for c in np.unique(codes)])
    var = (k / (k - 1)) * (resid_sums ** 2).sum() / (n ** 2)
    se = np.sqrt(var)
    tcrit = stats.t.ppf(1 - ALPHA / 2, df=k - 1)
    p = 2 * (1 - stats.t.cdf(abs(m / se), df=k - 1)) if se > 0 else np.nan
    return m, m - tcrit * se, m + tcrit * se, p, k

def bh_reject(pvals, q):
    """Benjamini-Hochberg. Returns a boolean array aligned to pvals."""
    p = np.asarray(pvals, float)
    ok = ~np.isnan(p)
    out = np.zeros(len(p), bool)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m = len(order)
    thresh = q * (np.arange(1, m + 1) / m)
    passed = p[order] <= thresh
    if passed.any():
        cut = np.max(np.where(passed)[0])
        out[order[:cut + 1]] = True
    return out

# (treatment label, treatment column, confounder list, contrasts, needs_trim)
SUBGROUP_TREATMENTS = [
    ("Gender",     "gender_num",      full_confounders,        [(0, 1)],         False),
    ("Location",   "location_num",    student_level_confounders, [(0, 1), (0, 2)], False),
    ("SchoolType", "school_type_num", student_level_confounders, [(0, 1)],         True),
]


In [ ]:
subgroup_rows = []
for tname, tcol, conf, comps, needs_trim in SUBGROUP_TREATMENTS:
    work = ghedudata.copy()
    n_before = len(work)
    if needs_trim:
        ov = LogisticRegression(max_iter=1000)
        ov.fit(work[conf].values, work[tcol].values)
        pr = ov.predict_proba(work[conf].values)[:, 1]
        keep = (pr >= TRIM_LO) & (pr <= TRIM_HI)
        work = work.loc[keep]
        print(f"\n[{tname}] FILTER overlap trim ({TRIM_LO}-{TRIM_HI}) "
              f"| n before {n_before} | n after {len(work)}")
    else:
        print(f"\n[{tname}] no trim declared | n = {len(work)}")

    X_raw = work[conf].values
    T = work[tcol].values
    Y = work["at_risk"].values
    Gv = work["School_ID"].values
    print(f"  X shape {X_raw.shape} | T levels {sorted(set(T))} | Y mean {Y.mean():.4f}")

    splitter = StratifiedGroupKFold(n_splits=N_SPLITS_SUBGROUP, shuffle=True, random_state=42)

    for T0, T1 in comps:
        contrast = f"{T1}v{T0}"
        oof = {arm: np.full(len(Y), np.nan) for arm in ("baseline", "engineered")}
        oof_w = {arm: np.full(len(Y), np.nan) for arm in ("baseline", "engineered")}
        n_skipped = 0
        for tr, te in splitter.split(X_raw, Y, groups=Gv):
            if len(set(T[tr])) < 2 or len(set(T[te])) < 2:
                n_skipped += len(te)
                continue
            sc = StandardScaler()
            X_tr, X_te = sc.fit_transform(X_raw[tr]), sc.transform(X_raw[te])
            for arm, leaf, cw in (("baseline", LEAF_BASELINE, None),
                                  ("engineered", LEAF_ENGINEERED, weight_from(Y[tr]))):
                m = fit_causal_forest(X_tr, T[tr], Y[tr], min_samples_leaf=leaf, class_weight=cw)
                oof[arm][te] = m.effect(X_te, T0=T0, T1=T1).ravel()
                lo, hi = m.effect_interval(X_te, T0=T0, T1=T1, alpha=ALPHA)
                oof_w[arm][te] = hi - lo
        covered = int((~np.isnan(oof["engineered"])).sum())
        print(f"  {contrast}: units with an out-of-fold estimate "
              f"| n before {len(Y)} | n after {covered} | skipped {n_skipped}")

        subgroup_defs = [("ALL UNITS", np.ones(len(Y), bool))]
        for fname in conf:
            for lvl in sorted(pd.unique(work[fname])):
                subgroup_defs.append((f"{fname}={lvl}", (work[fname].values == lvl)))
        print(f"  {contrast}: {len(subgroup_defs)} candidate subgroups from {len(conf)} confounders")

        for sname, mask in subgroup_defs:
            valid = mask & ~np.isnan(oof["engineered"]) & ~np.isnan(oof["baseline"])
            n_s = int(valid.sum())
            if n_s < MIN_SUBGROUP_N:
                subgroup_rows.append(dict(treatment=tname, contrast=contrast, subgroup=sname,
                                          n=n_s, arm="", note=f"below MIN_SUBGROUP_N={MIN_SUBGROUP_N}, not tested"))
                continue
            for arm in ("baseline", "engineered"):
                tau = oof[arm][valid]
                mean, lo, hi, p, k = cluster_robust_mean(tau, Gv[valid])
                subgroup_rows.append(dict(
                    treatment=tname, contrast=contrast, subgroup=sname,
                    arm=arm, n=n_s, n_schools=k,
                    cate_mean=mean, cr_lo=lo, cr_hi=hi, cr_p=p,
                    mean_unit_ci_width=float(np.nanmean(oof_w[arm][valid])),
                    note=""))

subgroup_res = pd.DataFrame(subgroup_rows)


In [ ]:
subgroup_res["cr_sig_neg_uncorrected"] = ((subgroup_res["cr_p"] < ALPHA) & (subgroup_res["cate_mean"] < 0))
subgroup_res["cr_sig_neg_bh"] = False
for keys, g in subgroup_res.dropna(subset=["cr_p"]).groupby(["treatment", "contrast", "arm"]):
    fam = g[g["subgroup"] != "ALL UNITS"]
    rej = bh_reject(fam["cr_p"].values, FDR_Q)
    subgroup_res.loc[fam.index, "cr_sig_neg_bh"] = rej & (fam["cate_mean"].values < 0)
subgroup_res.to_csv("subgroup_identification.csv", index=False)
print("WROTE subgroup_identification.csv |", len(subgroup_res), "rows")

print("\n" + "=" * 88)
print("OBJECTIVE 2 DELIVERABLE - subgroups with significantly negative CATE at 95%")
print("cluster-robust intervals, clustered on school | BH q =", FDR_Q)
print("=" * 88)
sig = subgroup_res[(subgroup_res["arm"] == "engineered") & subgroup_res["cr_sig_neg_bh"]]
if sig.empty:
    print("NO SUBGROUP reaches significance after correction.")
else:
    cols = ["treatment", "contrast", "subgroup", "n", "n_schools", "cate_mean", "cr_lo", "cr_hi", "cr_p"]
    print(sig[cols].sort_values("cate_mean").to_string(index=False))
n_unc = int(subgroup_res[(subgroup_res["arm"] == "engineered") & subgroup_res["cr_sig_neg_uncorrected"]].shape[0])
print(f"\nsubgroups significant UNCORRECTED : {n_unc}")
print(f"subgroups significant AFTER BH     : {int(sig.shape[0])}")

print("\n" + "=" * 88)
print("SIGNIFICANCE CROSSINGS: what does the narrower interval buy?")
print("=" * 88)
piv = subgroup_res[subgroup_res["subgroup"] != "ALL UNITS"].pivot_table(
    index=["treatment", "contrast", "subgroup"], columns="arm",
    values="cr_sig_neg_uncorrected", aggfunc="first")
if {"baseline", "engineered"}.issubset(piv.columns):
    piv = piv.dropna()
    gained = piv[(~piv["baseline"].astype(bool)) & piv["engineered"].astype(bool)]
    lost = piv[piv["baseline"].astype(bool) & (~piv["engineered"].astype(bool))]
    print(f"subgroups significant under engineered but NOT baseline: {len(gained)}")
    print(f"subgroups significant under baseline but NOT engineered: {len(lost)}")
    print(f"NET CROSSINGS: {len(gained) - len(lost)}")

print("\nMETHOD NOTE:")
print("  Cluster-robust intervals account for the 14-school structure but not")
print("  for the forest's own estimation uncertainty. Report both this and")
print("  that caveat together.")


## 11. Convergence Check (run last)


In [ ]:
report_convergence_log()